# Colab Pro+: approval-gated InternViT patch pilot

**One real subject, 200 training concepts, five epochs, seed 0, fair 200-way testing.** Three matched arms: original pooled route_c architecture, L28 mean control, and RPA-inspired L28 patch adapter. This is not an official RPA reproduction and does not claim to reproduce the historical 78.8% under its different protocol.

Pro+ does not guarantee a GPU model. Detect the actual accelerator and VRAM below. The user's last reported balance is **500 CU**; that is not a live balance. Colab's Resources panel is the authoritative source for available CU and current CU/hour. This notebook does not scrape private billing APIs or guess rates from GPU names. Enter those two observed values; GPU work remains blocked until approval.

Suggested pilot ceiling: **25 CU (5% of 500)** and **2 connected hours per session**, both adjustable before approval. These are budget limits, not predicted costs. Actual estimates follow a small extraction probe and one training epoch per arm. Estimated CU = observed CU/hour × connected hours. Idle/setup time also matters; logs are estimates, not billing statements.

Full ten-subject × three-seed training is not launched anywhere in this notebook. It requires a successful pilot and separate user approval.

## A. CPU-only pilot input preparation — run this first

Select **Runtime → Change runtime type → Hardware accelerator: None**, with High RAM if available. This phase downloads only sub-01 training/test EEG (~10.4 GB total). Do not allocate a GPU for downloads or verification. NPY dictionary loading can require about 18 GiB free RAM for the larger file; the verifier checks available RAM and loads files sequentially.

Existing image metadata, image paths and pooled InternViT cache shapes are inspected **before any dataset download**. No images, visual features, other subjects, or models are downloaded by this phase. Existing EEG files are size/hash-checked and never replaced. Interrupted downloads remain under `MyDrive/THINGS-EEG2/.pilot_downloads/` and resume on rerun. Byte identity is checked against the pinned Hub file's size/SHA-256 when supplied; actual shapes, dtype, finiteness and available channel/time metadata are recorded. Mirror origin is verified, but upstream whitening provenance remains insufficient for strict LOSO.

If images or visual caches are missing, the report lists them and leaves training blocked. Reuse existing image locations by editing IMAGE_ROOTS; do not download another image archive automatically.

**Dictionary-format fix / safe resume:** the default `VERIFY_EXISTING_EEG_ONLY = True` rechecks existing EEG without any download. Missing files or the saved download plan cause a clear stop. The fresh `/content/eeg-patch-pr2-cpu-dictfix` code checkout avoids reusing the earlier buggy code; it does not touch Drive EEG/cache paths. Both direct dictionary and scalar object-array formats are supported. For a genuinely missing file, the original download mode remains available only by explicitly setting this flag to false; verified existing files are still never redownloaded.


In [ ]:
import os, sys, subprocess, json, tempfile
from pathlib import Path
import torch
assert not torch.cuda.is_available(), 'Switch to a CPU-only runtime (accelerator None) for this preparation phase.'
from google.colab import drive
drive.mount('/content/drive')
REPO = Path(tempfile.mkdtemp(prefix='eeg-patch-pr2-cpu-dictfix-',dir='/content'))/'repo'
subprocess.run(['git','clone','--branch','audit/phase1-reproducible-alignment',
                'https://github.com/firozehrazavi/eeg-alignment-phase1.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface-hub==0.26.2','psutil==6.1.0','filelock==3.16.1'],check=True)
RAW = Path('/content/drive/MyDrive/THINGS-EEG2')
CACHE = Path('/content/drive/MyDrive/eeg_results/cache')
IMAGE_ROOTS = ['/content/drive/MyDrive/THINGS-EEG2','/content/drive/MyDrive/eeg_results','/content/EEG_Data']
cpu_env = dict(os.environ,CUDA_VISIBLE_DEVICES='')
cpu_args = [sys.executable,'-m','phase1.pilot_inputs','--root',str(RAW),'--cache',str(CACHE),
            '--image-roots',*IMAGE_ROOTS]
# Download is explicitly restricted to the two requested sub-01 files.
# The CLI itself runs the read-only image/cache inventory before contacting the Hub.
VERIFY_EXISTING_EEG_ONLY = True #@param {type:"boolean"}
mode = '--verify-existing-sub01' if VERIFY_EXISTING_EEG_ONLY else '--download-sub01'
subprocess.run(cpu_args+[mode],cwd=REPO,env=cpu_env,check=True)
pointer = json.loads((RAW/'pilot_inputs_latest.json').read_text())
report = json.loads(Path(pointer['report']).read_text())
print('Verification report:',pointer['report'])
print('EEG verification:',report['status'])
print('All image/pooled/EEG inputs ready:',report['ready_for_preparation'])
print('Missing:',report['inventory']['missing'])
assert report['ready_for_preparation'], 'EEG is retained; provide the missing image/cache inputs before continuing.'
WORK = Path('/content/drive/MyDrive/eeg_results/patch_phase1')
PREP = WORK/'pilot_sub01_200concepts_v1'
prepare_args = [sys.executable,'-m','phase1.patch_prepare','--root',str(RAW),'--cache',str(CACHE),
                '--image-root',report['inventory']['image_root'],'--output',str(PREP),
                '--subjects','sub-01','--concepts','200']
if PREP.exists():
    assert (PREP/'preparation_state.json').is_file(), 'Existing directory is preserved. Choose a new PREP path for verified CPU preparation.'
    prepare_args += ['--resume']
subprocess.run(prepare_args,cwd=REPO,env=cpu_env,check=True)
print('CPU preparation finished. STOP: review the report and obtain approval before allocating a GPU.')


## B. GPU pilot — separate approval required

Stop after section A. Do not switch runtimes until the pilot input report passes and GPU work is approved. After approval, switch to a GPU runtime and start **here**, skipping the CPU-only cell above. Downloads and preparation are not repeated. Actual CU balance/rate still come from Colab's Resources panel; all extraction/training approvals below remain false by default.

In [ ]:
APPROVE_GPU_SESSION = False #@param {type:"boolean"}
assert APPROVE_GPU_SESSION, 'Awaiting approval after CPU input verification.'
import time, os, json, subprocess, sys, uuid
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Connect a GPU runtime. No GPU model is guaranteed by Pro+.'
free,total = torch.cuda.mem_get_info()
GPU = {'name':torch.cuda.get_device_name(),'total_VRAM_GiB':total/2**30,'free_VRAM_GiB':free/2**30,
       'torch':torch.__version__,'cuda':torch.version.cuda,
       'precision':'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}
print(json.dumps(GPU,indent=2))
print('Last reported CU balance: 500; LIVE available CU and CU/hour: unknown until entered from Resources.')
# Colab: click the RAM/Disk resource indicator to view your balance and usage rate.
AVAILABLE_CU = 0.0 #@param {type:"number"}
CU_PER_HOUR = 0.0 #@param {type:"number"}
PILOT_CU_CAP = 25.0 #@param {type:"number"}
SESSION_HOUR_CAP = 2.0 #@param {type:"number"}
assert AVAILABLE_CU > 0 and CU_PER_HOUR > 0, 'Enter actual balance and CU/hour from Resources; do not guess.'
assert 0 < PILOT_CU_CAP <= AVAILABLE_CU and SESSION_HOUR_CAP > 0
print(f'Observed balance {AVAILABLE_CU:g} CU; observed rate {CU_PER_HOUR:g} CU/h.')
print(f'Pilot cap {PILOT_CU_CAP:g} CU; time allowed by cap {min(SESSION_HOUR_CAP,PILOT_CU_CAP/CU_PER_HOUR):.2f} h.')
SESSION_START = time.time()
from google.colab import drive
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/eeg_results/patch_phase1')
WORK.mkdir(parents=True,exist_ok=True)
LEDGER = WORK/'pilot_resource_usage.jsonl'
prior_rows = [json.loads(line) for line in LEDGER.read_text().splitlines() if line.strip()] if LEDGER.exists() else []
PRIOR_CU = max([r.get('estimated_cu') or 0 for r in prior_rows],default=0.)
assert PRIOR_CU < PILOT_CU_CAP, 'Previous pilot sessions reached the cap; review the ledger before increasing it.'
os.environ.update(PHASE1_SESSION_START=str(SESSION_START),PHASE1_CU_RATE=str(CU_PER_HOUR),
                  PHASE1_CU_LIMIT=str(PILOT_CU_CAP),PHASE1_CU_PRIOR=str(PRIOR_CU),
                  PHASE1_SECONDS_LIMIT=str(SESSION_HOUR_CAP*3600),PHASE1_RESOURCE_LOG=str(LEDGER))
SESSION = uuid.uuid4().hex[:8]
(WORK/f'runtime_{SESSION}.json').write_text(json.dumps(dict(**GPU,available_CU_observed=AVAILABLE_CU,
    CU_per_hour_observed=CU_PER_HOUR,observed_at=SESSION_START,prior_pilot_CU_estimate=PRIOR_CU),indent=2))

In [ ]:
REPO = Path('/content/eeg-patch-pr2-cpu-dictfix')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','audit/phase1-reproducible-alignment',
                    'https://github.com/firozehrazavi/eeg-alignment-phase1.git',str(REPO)],check=True)
os.chdir(REPO)
print('Pinned code:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
# Keep Colab's driver-compatible torch/torchvision, including on newer GPUs.
# Pin auxiliary dependencies and record the complete environment for replay.
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-patch-colab.txt'],check=True)
with open(WORK/f'environment_{SESSION}.txt','w') as f:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=f,check=True)
sys.path.insert(0,str(REPO))
from phase1.patch_resources import BudgetMeter, BudgetExceeded, runtime_profile
meter = BudgetMeter()
def monitored_run(args):
    meter.check('before_subprocess',force=True)
    child = subprocess.Popen(args,cwd=REPO,env=os.environ.copy())
    try:
        while True:
            try:
                result = child.wait(timeout=30)
                break
            except subprocess.TimeoutExpired:
                meter.check('subprocess_running',force=True)
        if result:
            raise RuntimeError(f'Child exited {result}; inspect its log. Completed cache rows/checkpoints remain in Drive.')
    except BaseException:
        child.terminate()
        try: child.wait(timeout=10)
        except subprocess.TimeoutExpired: child.kill(); child.wait()
        raise
    finally:
        meter.check('after_subprocess',force=True)
print(runtime_profile())
monitored_run([sys.executable,'-m','unittest','discover','-s','tests','-v'])
# Tests are software validation only. They never substitute for real EEG.

## Real-data preflight and preparation

No real data is bundled. The paths below must exist in your Drive; the notebook reports exact missing files. Distributed preprocessed EEG is labeled `benchmark_preprocessed`, because its upstream whitening has not been audited for strict induction. `_z16.npy` is not accepted as a substitute. Input pickle files must come from a trusted dataset source.

Preparation selects concepts deterministically, averages repetitions, retains the 17 posterior channels and all supplied time samples, verifies original image hashes and preserves old caches. If preparation fails halfway, choose a new directory rather than overwriting it.

In [ ]:
RAW = Path('/content/drive/MyDrive/THINGS-EEG2')
CACHE = Path('/content/drive/MyDrive/eeg_results/cache')
PREP = WORK/'pilot_sub01_200concepts_v1'
pointer = json.loads((RAW/'pilot_inputs_latest.json').read_text())
report = json.loads(Path(pointer['report']).read_text())
assert report['status']=='eeg_verified' and report['ready_for_preparation'], 'Complete CPU input verification first.'
assert (PREP/'dataset.json').is_file(), 'Complete the CPU-only preparation stage before GPU work.'
print('CPU report:',pointer['report'])
print('Prepared dataset:',PREP/'dataset.json')


## Approval 1: small extraction probe

The first probe loads the frozen backbone and extracts **32 images only**. This incurs GPU time and a model download; approve it explicitly. A single native FP16 layer needs approximately 13.4 GiB for the entire pilot, versus 102.2 GiB for the full dataset. Model weights and local staging need additional disk. The extractor starts at batch 1, uses BF16 where supported and FP16 otherwise, captures one block and discards CLS. If the weights cannot fit the assigned GPU, it stops and reports that limitation; it does not change the backbone.

Completion masks permit extraction to resume. The pinned model revision and extraction precision must match when resuming. Changing GPU precision requires a separate cache to preserve comparability.

In [ ]:
APPROVE_EXTRACTION_PROBE = False #@param {type:"boolean"}
assert APPROVE_EXTRACTION_PROBE, 'Awaiting approval for the 32-image extraction probe.'
from huggingface_hub import model_info
revision_file = PREP/'internvit_revision.txt'
if not revision_file.exists():
    revision_file.write_text(model_info('OpenGVLab/InternViT-6B-448px-V2_5').sha)
revision = revision_file.read_text().strip()
TOKEN_DIR = PREP/'tokens_L28'
extract_args = [sys.executable,'-m','phase1.patch_cache','--catalog',str(PREP/'catalog.json'),
                '--output',str(TOKEN_DIR),'--revision',revision,'--layer','28','--batch','1']
if not (TOKEN_DIR/'throughput.json').exists():
    monitored_run(extract_args+['--limit','32'])
timing = json.loads((TOKEN_DIR/'throughput.json').read_text())
catalog = json.loads((PREP/'catalog.json').read_text())
import numpy as np
remaining = sum(len(catalog[s])-int(np.load(TOKEN_DIR/f'{s}_done.npy').sum())
                if (TOKEN_DIR/f'{s}_done.npy').exists() else len(catalog[s]) for s in ['train','test'])
estimated_extract_hours = remaining*timing['seconds_per_image']/3600
estimated_extract_cu = estimated_extract_hours*CU_PER_HOUR
print(f'Remaining extraction estimate: {estimated_extract_hours:.2f} h, {estimated_extract_cu:.2f} CU.')
print('Planning allowance (+50%):',round(1.5*estimated_extract_cu,2),'CU; excludes another model load/download.')
print('Training cost is not known yet; it will be measured before the five-epoch pilot.')
meter.check('extraction_probe_complete',force=True)

In [ ]:
APPROVE_REMAINING_EXTRACTION = False #@param {type:"boolean"}
assert APPROVE_REMAINING_EXTRACTION, 'Review the estimate above before approving extraction.'
spent = meter.check('extraction_approval',force=True)['estimated_cu']
assert spent+1.5*estimated_extract_cu < PILOT_CU_CAP, 'Estimate exceeds remaining pilot allowance; review budget.'
monitored_run(extract_args)
# Stage completed arrays on local disk for training; keep original caches in Drive.
import shutil
LOCAL = Path('/content/patch_pilot_data')
if not LOCAL.exists():
    needed = sum(p.stat().st_size for p in PREP.rglob('*') if p.is_file())
    assert shutil.disk_usage('/content').free > needed*1.1, 'Insufficient local staging space.'
    shutil.copytree(PREP,LOCAL)
else:
    # On reconnect, only reuse a previously completed local staging directory.
    assert (LOCAL/'STAGING_COMPLETE').is_file(), 'Choose a new LOCAL path after incomplete staging.'
(LOCAL/'STAGING_COMPLETE').write_text('Staged from '+str(PREP))
from phase1.patch_cache import TokenCache
TokenCache(LOCAL/'tokens_L28',LOCAL/'catalog.json')
meter.check('staging_complete',force=True)

## Approval 2: one epoch per arm, no test evaluation

This measures training speed with the **final five-epoch LR schedule**. All arms use the same EEG initialization, contrastive batch 64, seed, split and normalization. BF16/FP16 is selected from the actual GPU; contrastive logits and retrieval scores remain FP32. The adapter streams CPU token chunks to GPU, with activation checkpointing. OOM retries halve the adapter chunk, never the contrastive batch. If chunk 1 still fails, stop and use a suitable runtime or a newly declared matched configuration.

Drive checkpoints save model, optimizer, scheduler, AMP scaler and RNG states at epoch boundaries. The next step resumes these exact checkpoints; interrupted epochs replay from the last saved epoch. Test labels are not evaluated in the profiling stage.

In [ ]:
APPROVE_TRAINING_PROFILE = False #@param {type:"boolean"}
assert APPROVE_TRAINING_PROFILE, 'Awaiting approval for one training epoch per arm.'
OUT = WORK/'pilot_sub01_200concepts_run1'
train_args = [sys.executable,'-m','phase1.patch_experiment','--dataset',str(LOCAL/'dataset.json'),
              '--config',str(REPO/'configs/patch_pilot.json'),'--output',str(OUT)]
if not (OUT/'PILOT_PROFILE_COMPLETE').exists():
    monitored_run(train_args+['--profile-one-epoch']+(['--resume'] if OUT.exists() else []))
timings = [json.loads(line) for line in (OUT/'epoch_timing.jsonl').read_text().splitlines()]
per_arm = {}
for row in timings: per_arm[row['run']] = row['seconds']
assert len(per_arm)==3, 'Need a timing measurement for every arm.'
remaining_train_hours = 4*sum(per_arm.values())/3600
estimated_train_cu = remaining_train_hours*CU_PER_HOUR
print('Epoch seconds per arm:',per_arm)
print(f'Remaining four epochs/arm: {remaining_train_hours:.2f} h, {estimated_train_cu:.2f} CU estimate.')
print('With +50% planning allowance:',round(estimated_train_cu*1.5,2),'CU, plus loading/evaluation overhead.')
print('Measured peak GPU allocation GiB:',json.loads((OUT/'provenance.json').read_text())['peak_gpu_allocated_bytes']/2**30)
meter.check('training_profile_complete',force=True)

In [ ]:
APPROVE_FIVE_EPOCH_PILOT = False #@param {type:"boolean"}
assert APPROVE_FIVE_EPOCH_PILOT, 'Review cost/memory measurements and approve before continuing.'
spent = meter.check('pilot_approval',force=True)['estimated_cu']
assert spent+estimated_train_cu*1.5 < PILOT_CU_CAP, 'Estimate exceeds remaining allowance; review budget.'
monitored_run(train_args+['--resume'])
assert (OUT/'COMPLETE').is_file()
print((OUT/'results.csv').read_text())
meter.check('pilot_complete',force=True)
print('Compare the estimate to Resources now. No full training was launched.')
print('Disconnect/delete the runtime after saving to avoid idle CU consumption.')

## After the pilot — stop here

The full configurations are prepared but **not launched**. Review all three arms, 200-way Top-1/Top-5, measured runtime/VRAM, cost ledger and failures before asking for approval to run ten subjects × three seeds. Keep future full within-subject, zero-calibration LOSO, strict inductive LOSO, CSLS transduction and labeled calibration results separate.

The distributed EEG pilot is `benchmark_preprocessed`, not strict induction. Strict mode requires audited target-independent preprocessing, not merely a changed flag. Multi-positive alignment is a separate objective ablation; text supervision and SCORE are not included here.

Cost logs estimate connected time from the entered rate. They cannot read live billing, include other runtimes, account for unobserved rate changes, or guarantee an exact spending cap. Recheck Resources when reconnecting, update the rate/balance, and inspect `pilot_resource_usage.jsonl`. The next session starts a new clock and carries forward the previous estimated pilot spend. The supervisor checks every 30 seconds, the trainer/extractor check batch boundaries, and completed cache rows/epoch checkpoints survive a budget stop. **A stopped cell does not disconnect the GPU runtime.** Disconnect it manually after stopping, or use the optional cell below.

If a Pro+ session continues in the background, the same budget limits remain active; leaving the browser is not approval to launch further experiments. The original notebooks and historical results remain untouched.

In [ ]:
DISCONNECT_RUNTIME_NOW = False #@param {type:"boolean"}
if DISCONNECT_RUNTIME_NOW:
    from google.colab import runtime
    runtime.unassign()